pre ove sveski **OBAVEZNO** pokrenuti features.ipynb 

In [180]:
import pandas as pd

In [181]:
import warnings
warnings.filterwarnings("ignore")

In [182]:
#ucitavanje neophodnih tabela
pits = pd.read_csv('tables//all_pit_stops1.csv')
all_data = pd.read_csv('tables//trke_sa_driverId.csv')

In [183]:
all_data = all_data.drop('_merge', axis=1)

In [184]:
all_data.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', 'AirTemp', 'Humidity',
       'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin', 'wind_direction',
       'is_sprint_weekend', 'avg_position_last5', 'num_dnfs_last5',
       'avg_gained_lost_last5'],
      dtype='object')

In [185]:
pits.columns

Index(['season', 'round', 'race_name', 'driverId', 'lap', 'stop', 'duration',
       'time'],
      dtype='object')

In [186]:
pits = pits.drop('time', axis=1)

In [187]:
#izostavljanje nepotrebnih sezona
pits = pits[pits['season'] != 2015]
pits = pits[pits['season'] != 2016]
pits = pits[pits['season'] != 2017]

In [188]:
#koje su sezone u tabelama
seasons = list(set(zip(pits['season'])))

In [189]:
#posto prethodna linija daje listu vrednosti oblika [(1,),(2.)], ovaj kod pretvara u listu [1, 2]
for i in range(len(seasons)):
    seasons[i] = (seasons[i][0])
seasons = sorted(seasons)

In [190]:
print(seasons)

[2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [191]:
# koliko rundi ima po sezoni
num_rounds = []
for season in seasons:
    #posto prethodna linija daje listu vrednosti oblika [(1,),(2.)], ovaj kod pretvara u listu [1, 2]
    one_season_rounds = pits[pits['season'] == season]
    rounds = list(set(zip(one_season_rounds['round'])))
    for i in range(len(rounds)):
        rounds[i] = (rounds[i][0])
    num_rounds.append(len(rounds))
    # print(num_rounds)
    

    

In [192]:
df = pd.DataFrame(columns=['season', 'round', 'driverId', 'num_pit_stops', 'lap_pit_stops', 'duration_pit_stops'])
for i in range(len(seasons)):
    for j in range(num_rounds[i]):
        one_race = pits[(pits['season'] == seasons[i]) & (pits['round'] == j+1)]
        #ovde se izracunavaju broj stajanja, duzina svakog i u kom krugu se odigrala
        season = seasons[i]
        round = j+1
        #print(one_race)
        one_race = one_race.drop(['season', 'round', 'race_name'], axis=1)
        # mekes dictonary example: 'alonso': [{'lap': 26, 'stop': 1, 'duration': '22.573'}] 
        pit_stops_dict = one_race.groupby('driverId').apply(lambda g: g.drop(columns='driverId').to_dict(orient='records')).to_dict()
        for driver in pit_stops_dict:
            one_driver = pit_stops_dict[driver]
            num = len(one_driver)
            lap, duration = [], []
            for x in one_driver:
                lap.append(x['lap'])
                duration.append(x['duration'])
            df.loc[len(df)] = [season, round, driver, num, lap, duration]

In [193]:
all_data = pd.merge(all_data, df, on=['season', 'round', 'driverId'], how='left')        

In [198]:
len(all_data)

2979

season                                   2024
round                                      24
drivers_num                                11
position_quali                             10
position_race                              20
status                              Collision
driver                                  Pérez
constructor                          Red Bull
grid                                       10
laps                                        0
total_time_ms                             NaN
points                                    0.0
driverId                                perez
race_name                Abu Dhabi Grand Prix
circuit                    Yas Marina Circuit
 lap_length                             5.281
 number_of_laps                          58.0
 number_of_corners                       16.0
AirTemp                             26.768243
Humidity                            51.445946
Pressure                          1017.426351
TrackTemp                         